# Lenguajes y Autómatas II (SCD-1016)
## Sesión 16 · Firmas de Función y Parámetros Formales

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo comprueba el compilador que llamaste a una función con los argumentos correctos?
>
> **Objetivo de la sesión:** Registrar firmas con nombre y aridad, y verificar cada llamada contra ese registro devolviendo el diagnóstico adecuado.

**Entregable:** tu función verificar_llamada() en artesano_sesion16_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion16.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes un catálogo de firmas ya registradas y una llamada descrita por nombre y número de argumentos; busca el nombre y compara la aridad. La función devuelve 1 si la llamada es válida, 0 si la aridad no coincide y -1 si el nombre no existe.

Al final de la sesión tendrás: **Un verificador de firmas que distingue llamada válida, aridad incorrecta y función inexistente.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 16](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion16.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion16.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion16.h"

int verificar_llamada(const Funcion *catalogo, int total, const char *nombre, int nargs);
```

- La búsqueda del nombre es exacta y recorre el catálogo completo.
- Una función registrada con cero parámetros solo acepta cero argumentos.
- Un catálogo NULL, un nombre NULL o una aridad negativa devuelven -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion16_c.c` junto con `artesano_sesion16_c.c`
> (o `ia_sesion16_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `suma con 3 argumentos` | 1 |
| 2 | `suma con 2 argumentos` | 0 |
| 3 | `suma con 4 argumentos` | 0 |
| 4 | `funcion no registrada` | -1 |
| 5 | `main sin argumentos` | 1 |
| 6 | `nombre NULL` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 16 - Registro de prototipos y aridad ===
Caso                               esperado:              obtenido:              resultado
[ 1] suma con 3 argumentos              esperado: 1            obtenido: 1            OK
[ 2] suma con 2 argumentos              esperado: 0            obtenido: 0            OK
[ 3] suma con 4 argumentos              esperado: 0            obtenido: 0            OK
[ 4] funcion no registrada              esperado: -1           obtenido: -1           OK
[ 5] main sin argumentos                esperado: 1            obtenido: 1            OK
[ 6] nombre NULL                        esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion16.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] La aridad declarada se compara con la aridad de la llamada, no con el número de firmas del catálogo.
- [ ] Una función con cero parámetros se acepta únicamente con cero argumentos.
- [ ] Un nombre inexistente devuelve -1 sin confundirse con una aridad incorrecta.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion16_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion16.h
/* api_sesion16.h - contrato de la Sesion 16
 * Registro de prototipos y aridad
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION16_H
#define API_SESION16_H

typedef struct Funcion {
    const char *nombre;
    int aridad;
} Funcion;

int verificar_llamada(const Funcion *catalogo, int total, const char *nombre, int nargs);

#endif /* API_SESION16_H */


Writing api_sesion16.h


In [3]:
%%writefile artesano_sesion16_c.c
/* artesano_sesion16_c.c - Fase 1: El Artesano (sin IA)
 * Registro de prototipos y aridad.
 * No lleva main(), el main es del harness.
 */
#include "api_sesion16.h"
#include <stdio.h>
#include <string.h>

int verificar_llamada(const Funcion *catalogo, int total, const char *nombre, int nargs) {
    // primero lo invalido
    if (catalogo == NULL || nombre == NULL || nargs < 0 || total < 0) return -1;

    // recorro TODO el catalogo buscando el nombre
    for (int i = 0; i < total; i++) {
        if (catalogo[i].nombre != NULL && strcmp(catalogo[i].nombre, nombre) == 0) {
            // ya la encontre, ahora checo los argumentos
            if (catalogo[i].aridad == nargs) return 1; // llamada bien
            return 0; // numero de argumentos mal
        }
    }
    return -1; // esa funcion no existe
}

Writing artesano_sesion16_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion16_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


### Fase 2 · El Científico — Contrato C-R-E-O

**C - Contexto:**
En Lenguajes y Autómatas II estamos viendo cómo el compilador revisa las llamadas a funciones. Cada función se registra en un catálogo con su nombre y su aridad (cuántos parámetros recibe). Cuando el programa llama a una función, el compilador busca su nombre y revisa que el número de argumentos sea igual a la aridad.
Es en C (C11) y se compila con `-Wall -Wextra -Werror`, sin warnings. Solo librerías estándar y sin `main()`, porque el harness ya trae uno. El struct `Funcion` y la firma están en `api_sesion16.h` y no se cambian.

**R - Rol:**
Que la IA actúe como ingeniero de software de sistemas y compiladores que sabe cómo se verifican los prototipos de funciones.

**E - Especificación:**
`int verificar_llamada(const Funcion *catalogo, int total, const char *nombre, int nargs);`
- Busca el nombre en **todo** el catálogo (de 0 a `total-1`), con comparación exacta (`strcmp`). No basta con ver el primero.
- Si lo encuentra y `aridad == nargs`, regresa 1 (llamada válida).
- Si lo encuentra pero la aridad no coincide, regresa 0.
- Si no lo encuentra en ningún lado, regresa -1.
- Una función con aridad 0 solo acepta 0 argumentos.
- Hay que comparar `nargs` contra la aridad de la función, **no** contra `total`.
- Si `catalogo` es NULL, `nombre` es NULL o `nargs` es negativo, regresa -1.

Casos que debe pasar:
| Caso | Esperado |
| :--- | :--- |
| suma (aridad 3) con 3 argumentos | 1 |
| suma con 2 argumentos | 0 |
| suma con 4 argumentos | 0 |
| resta (no está registrada) | -1 |
| main (aridad 0) con 0 argumentos | 1 |
| nombre NULL | -1 |

**O - Output:**
Solo el código de `ia_sesion16_c.c`, corto, sin `main()` y sin explicación larga. Al final una línea con la complejidad.

**Lo que revisé antes de compilar:**
Que no cambiara la firma, que no trajera `main()`, que recorriera todo el catálogo y que comparara contra la aridad y no contra `total`. Luego lo probé con un catálogo de varias funciones, porque el harness solo usa una.

In [4]:
%%writefile ia_sesion16_c.c
/* ia_sesion16_c.c - Fase 2: El Cientifico
 * Codigo que me dio la IA (misma firma, sin main).
 */
#include "api_sesion16.h"
#include <string.h>

int verificar_llamada(const Funcion *catalogo, int total, const char *nombre, int nargs) {
    if (!catalogo || !nombre || nargs < 0 || total < 0) return -1;

    for (int i = 0; i < total; i++) {
        if (catalogo[i].nombre && strcmp(catalogo[i].nombre, nombre) == 0) {
            return catalogo[i].aridad == nargs ? 1 : 0;
        }
    }
    return -1; // no esta registrada
}

// Complejidad: O(n), n = funciones en el catalogo


Writing ia_sesion16_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion16_c.c
/* harness_sesion16_c.c - auditoria automatica de la Sesion 16
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion16_c.c artesano_sesion16_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion16.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 16 - Registro de prototipos y aridad ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Funcion c[] = {{"suma", 3}}; int r = verificar_llamada(c, 1, "suma", 3); verificar_int("suma con 3 argumentos", (long)(1), (long)(r)); }
    { Funcion c[] = {{"suma", 3}}; int r = verificar_llamada(c, 1, "suma", 2); verificar_int("suma con 2 argumentos", (long)(0), (long)(r)); }
    { Funcion c[] = {{"suma", 3}}; int r = verificar_llamada(c, 1, "suma", 4); verificar_int("suma con 4 argumentos", (long)(0), (long)(r)); }
    { Funcion c[] = {{"suma", 3}}; int r = verificar_llamada(c, 1, "resta", 2); verificar_int("funcion no registrada", (long)(-1), (long)(r)); }
    { Funcion c[] = {{"main", 0}}; int r = verificar_llamada(c, 1, "main", 0); verificar_int("main sin argumentos", (long)(1), (long)(r)); }
    { Funcion c[] = {{"suma", 3}}; int r = verificar_llamada(c, 1, NULL, 3); verificar_int("nombre NULL", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion16.md" "\n");
    return 0;
}


Writing harness_sesion16_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion16_c.c artesano_sesion16_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 16 - Registro de prototipos y aridad ===
Caso                               esperado:              obtenido:              resultado
[ 1] suma con 3 argumentos              esperado: 1            obtenido: 1            OK
[ 2] suma con 2 argumentos              esperado: 0            obtenido: 0            OK
[ 3] suma con 4 argumentos              esperado: 0            obtenido: 0            OK
[ 4] funcion no registrada              esperado: -1           obtenido: -1           OK
[ 5] main sin argumentos                esperado: 1            obtenido: 1            OK
[ 6] nombre NULL                        esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion16.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion16_c.c ia_sesion16_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 16 - Registro de prototipos y aridad ===
Caso                               esperado:              obtenido:              resultado
[ 1] suma con 3 argumentos              esperado: 1            obtenido: 1            OK
[ 2] suma con 2 argumentos              esperado: 0            obtenido: 0            OK
[ 3] suma con 4 argumentos              esperado: 0            obtenido: 0            OK
[ 4] funcion no registrada              esperado: -1           obtenido: -1           OK
[ 5] main sin argumentos                esperado: 1            obtenido: 1            OK
[ 6] nombre NULL                        esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion16.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion16.md` a `auditorias/auditoria_sesion16.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
